# Analyze per-query retrieval improvements

This notebook compares the saved per-query ranks of the base and fine-tuned BGE-M3 models. It identifies questions where fine-tuning moved the correct answer substantially higher, shows the question and its reference answer, and saves the analysis for thesis review.

Run `evaluate_porseman_models.ipynb` first. This notebook does not load a model or use the GPU.

## 1. Configure report inputs

The default labels match the four-model evaluation notebook. Change only the two labels below if a previous evaluation used different filenames.

In [ ]:
import csv
import html
import json
from pathlib import Path

from IPython.display import HTML, display

def find_project_root(start_path):
    for candidate in (start_path, *start_path.parents):
        if (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not find the project root. Run this notebook from inside the repository.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
REPORT_DIR = PROJECT_ROOT / 'reports/evaluations/porseman_model_comparison'
TEST_PATH = PROJECT_ROOT / 'data/processed/porseman_test.csv'
OUTPUT_PATH = REPORT_DIR / 'per_query_improvement_base_vs_fine_tuned.csv'

BASE_LABEL = 'base_bge_m3'
FINE_TUNED_LABEL = 'fine_tuned_bge_m3'
SAMPLE_SIZE = 20
MIN_RANK_IMPROVEMENT = 5
RETRIEVED_TOP_K = 5

BASE_RANK_PATH = REPORT_DIR / f'per_query_ranks_{BASE_LABEL}.csv'
FINE_TUNED_RANK_PATH = REPORT_DIR / f'per_query_ranks_{FINE_TUNED_LABEL}.csv'
BASE_RETRIEVAL_PATH = REPORT_DIR / f'per_query_top_{RETRIEVED_TOP_K}_{BASE_LABEL}.jsonl'
FINE_TUNED_RETRIEVAL_PATH = REPORT_DIR / f'per_query_top_{RETRIEVED_TOP_K}_{FINE_TUNED_LABEL}.jsonl'

print(f'Base ranks: {BASE_RANK_PATH}')
print(f'Fine-tuned ranks: {FINE_TUNED_RANK_PATH}')
print(f'Base retrieved answers: {BASE_RETRIEVAL_PATH}')
print(f'Fine-tuned retrieved answers: {FINE_TUNED_RETRIEVAL_PATH}')
print(f'Test data: {TEST_PATH}')

## 2. Load and validate the saved ranks

`rank_improvement` is `base_rank - fine_tuned_rank`. A positive value means that the fine-tuned model ranks the correct answer higher.

In [ ]:
def read_csv_rows(path, required_fields):
    if not path.is_file():
        raise FileNotFoundError(f'File not found: {path}. Run the evaluation notebook first or update the label.')
    with path.open(encoding='utf-8-sig', newline='') as source:
        reader = csv.DictReader(source)
        missing = set(required_fields) - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
        rows = list(reader)
    if len({row['id'] for row in rows}) != len(rows):
        raise ValueError(f'{path.name} contains duplicate IDs.')
    return rows

def read_retrieval_rows(path):
    if not path.is_file():
        raise FileNotFoundError(
            f'File not found: {path}. Re-run evaluate_porseman_models.ipynb to save the retrieved answers.'
        )
    with path.open(encoding='utf-8') as source:
        rows = [json.loads(line) for line in source if line.strip()]
    required_fields = {'id', 'correct_answer_id', 'correct_answer_rank', 'retrieved'}
    if any(required_fields - set(row) for row in rows):
        raise ValueError(f'{path.name} has an invalid retrieval record.')
    if len({row['id'] for row in rows}) != len(rows):
        raise ValueError(f'{path.name} contains duplicate query IDs.')
    return rows

rank_fields = {'id', 'question', 'correct_answer_rank'}
test_fields = {'id', 'question', 'content_text'}
base_rows = read_csv_rows(BASE_RANK_PATH, rank_fields)
fine_tuned_rows = read_csv_rows(FINE_TUNED_RANK_PATH, rank_fields)
base_retrieval_rows = read_retrieval_rows(BASE_RETRIEVAL_PATH)
fine_tuned_retrieval_rows = read_retrieval_rows(FINE_TUNED_RETRIEVAL_PATH)
raw_test_rows = read_csv_rows(TEST_PATH, test_fields)
seen_answers = set()
test_rows = []
for row in raw_test_rows:
    if row['content_text'] not in seen_answers:
        seen_answers.add(row['content_text'])
        test_rows.append(row)

base_by_id = {row['id']: row for row in base_rows}
fine_tuned_by_id = {row['id']: row for row in fine_tuned_rows}
base_retrieval_by_id = {row['id']: row for row in base_retrieval_rows}
fine_tuned_retrieval_by_id = {row['id']: row for row in fine_tuned_retrieval_rows}
test_by_id = {row['id']: row for row in test_rows}

if set(base_by_id) != set(fine_tuned_by_id):
    raise ValueError('Base and fine-tuned rank files do not contain the same query IDs.')
if set(base_by_id) != set(base_retrieval_by_id) or set(base_by_id) != set(fine_tuned_retrieval_by_id):
    raise ValueError('Rank files and retrieved-answer files do not contain the same query IDs.')
if set(base_by_id) != set(test_by_id):
    raise ValueError('Rank files and the deduplicated test data do not contain the same query IDs. Evaluate the fixed test split again.')

comparison_rows = []
for row_id, base_row in base_by_id.items():
    fine_tuned_row = fine_tuned_by_id[row_id]
    test_row = test_by_id[row_id]
    base_rank = int(base_row['correct_answer_rank'])
    fine_tuned_rank = int(fine_tuned_row['correct_answer_rank'])
    comparison_rows.append({
        'id': row_id,
        'question': test_row['question'],
        'correct_answer': test_row['content_text'],
        'base_rank': base_rank,
        'fine_tuned_rank': fine_tuned_rank,
        'rank_improvement': base_rank - fine_tuned_rank,
        'base_recall_at_1': int(base_rank <= 1),
        'fine_tuned_recall_at_1': int(fine_tuned_rank <= 1),
        'recall_at_1_gain': int(base_rank > 1 and fine_tuned_rank <= 1),
        'base_recall_at_5': int(base_rank <= 5),
        'fine_tuned_recall_at_5': int(fine_tuned_rank <= 5),
        'recall_at_5_gain': int(base_rank > 5 and fine_tuned_rank <= 5),
        'base_reciprocal_rank_at_10': 1.0 / base_rank if base_rank <= 10 else 0.0,
        'fine_tuned_reciprocal_rank_at_10': 1.0 / fine_tuned_rank if fine_tuned_rank <= 10 else 0.0,
        'mrr_at_10_gain': (1.0 / fine_tuned_rank if fine_tuned_rank <= 10 else 0.0) - (1.0 / base_rank if base_rank <= 10 else 0.0),
        'entered_top_10': int(base_rank > 10 and fine_tuned_rank <= 10),
    })

comparison_rows.sort(key=lambda row: (-row['rank_improvement'], row['fine_tuned_rank'], row['id']))
print(f'Compared queries: {len(comparison_rows):,}')

## 3. Summarize and save all per-query changes

The saved CSV contains every query, not only positive improvements. It can be used later for detailed thesis analysis.

In [ ]:
improved_rows = [row for row in comparison_rows if row['rank_improvement'] > 0]
unchanged_rows = [row for row in comparison_rows if row['rank_improvement'] == 0]
regressed_rows = [row for row in comparison_rows if row['rank_improvement'] < 0]

print(f'Improved queries: {len(improved_rows):,} ({len(improved_rows) / len(comparison_rows):.2%})')
print(f'Unchanged queries: {len(unchanged_rows):,} ({len(unchanged_rows) / len(comparison_rows):.2%})')
print(f'Regressed queries: {len(regressed_rows):,} ({len(regressed_rows) / len(comparison_rows):.2%})')
print(f'Recall@1 gains: {sum(row["recall_at_1_gain"] for row in comparison_rows):,}')
print(f'Recall@5 gains: {sum(row["recall_at_5_gain"] for row in comparison_rows):,}')
print(f'MRR@10 gains: {sum(row["mrr_at_10_gain"] > 0 for row in comparison_rows):,}')
print(f'Mean MRR@10 gain per query: {sum(row["mrr_at_10_gain"] for row in comparison_rows) / len(comparison_rows):.4f}')
print(f'Entered top 10: {sum(row["entered_top_10"] for row in comparison_rows):,}')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('w', encoding='utf-8-sig', newline='') as target:
    writer = csv.DictWriter(target, fieldnames=list(comparison_rows[0]))
    writer.writeheader()
    writer.writerows(comparison_rows)

print(f'All per-query changes: {OUTPUT_PATH}')

## 4. Inspect the strongest improvements

These are the most useful candidates for qualitative analysis in the thesis. The correct answer is shown so that the semantic relation can be reviewed directly.

In [ ]:
def short(text, limit=700):
    text = ' '.join((text or '').split())
    return text if len(text) <= limit else text[:limit].rstrip() + ' ...'

def render_retrieved(query_id, retrieval_record):
    items = []
    for item in retrieval_record['retrieved']:
        answer_id = item['answer_id']
        answer = test_by_id[answer_id]['content_text']
        marker = ' <b>(correct answer)</b>' if answer_id == query_id else ''
        items.append(
            f'<li><small>#{item["rank"]} | score={item["score"]:.4f} | {html.escape(answer_id)}</small><br>'
            f'{html.escape(short(answer, 500))}{marker}</li>'
        )
    return '<ol>' + ''.join(items) + '</ol>'

def display_examples(rows, title):
    table_rows = [
        '<tr><th>ID</th><th>Base rank</th><th>Fine-tuned rank</th><th>Improvement</th><th>Question</th><th>Correct answer</th><th>Base top 5</th><th>Fine-tuned top 5</th></tr>'
    ]
    for row in rows:
        table_rows.append(
            '<tr>'
            f'<td>{html.escape(row["id"])}</td>'
            f'<td>{row["base_rank"]:,}</td>'
            f'<td>{row["fine_tuned_rank"]:,}</td>'
            f'<td>{row["rank_improvement"]:+,}</td>'
            f'<td>{html.escape(short(row["question"], 350))}</td>'
            f'<td>{html.escape(short(row["correct_answer"], 700))}</td>'
            f'<td>{render_retrieved(row["id"], base_retrieval_by_id[row["id"]])}</td>'
            f'<td>{render_retrieved(row["id"], fine_tuned_retrieval_by_id[row["id"]])}</td>'
            '</tr>'
        )
    display(HTML(
        f'<h4>{html.escape(title)}</h4>'
        "<div style='overflow-x:auto'><table style='width:100%;text-align:right;border-collapse:collapse'>"
        + ''.join(table_rows) + '</table></div>'
    ))

def display_metric_examples(rows, title, base_key, fine_tuned_key, gain_key, percent=False):
    table_rows = [
        '<tr><th>ID</th><th>Base</th><th>Fine-tuned</th><th>Gain</th><th>Base rank</th><th>Fine-tuned rank</th><th>Question</th><th>Correct answer</th><th>Base top 5</th><th>Fine-tuned top 5</th></tr>'
    ]
    for row in rows:
        formatter = (lambda value: f'{value:.4f}') if percent else (lambda value: str(value))
        gain = row[gain_key]
        gain_text = f'{gain:+.4f}' if percent else f'{gain:+d}'
        table_rows.append(
            '<tr>'
            f'<td>{html.escape(row["id"])}</td>'
            f'<td>{formatter(row[base_key])}</td>'
            f'<td>{formatter(row[fine_tuned_key])}</td>'
            f'<td>{gain_text}</td>'
            f'<td>{row["base_rank"]:,}</td>'
            f'<td>{row["fine_tuned_rank"]:,}</td>'
            f'<td>{html.escape(short(row["question"], 350))}</td>'
            f'<td>{html.escape(short(row["correct_answer"], 700))}</td>'
            f'<td>{render_retrieved(row["id"], base_retrieval_by_id[row["id"]])}</td>'
            f'<td>{render_retrieved(row["id"], fine_tuned_retrieval_by_id[row["id"]])}</td>'
            '</tr>'
        )
    display(HTML(
        f'<h4>{html.escape(title)}</h4>'
        "<div style='overflow-x:auto'><table style='width:100%;text-align:right;border-collapse:collapse'>"
        + ''.join(table_rows) + '</table></div>'
    ))

strongest_improvements = [
    row for row in comparison_rows
    if row['rank_improvement'] >= MIN_RANK_IMPROVEMENT
][:SAMPLE_SIZE]

display_examples(
    strongest_improvements,
    f'Strongest improvements (at least {MIN_RANK_IMPROVEMENT} rank positions)',
)

## 5. Inspect queries improved for Recall and MRR

The following three views directly answer which queries improved for `Recall@1`, `Recall@5`, and `MRR@10`.

In [ ]:
recall_at_1_gain_rows = [row for row in comparison_rows if row['recall_at_1_gain']][:SAMPLE_SIZE]
recall_at_5_gain_rows = [row for row in comparison_rows if row['recall_at_5_gain']][:SAMPLE_SIZE]
mrr_at_10_gain_rows = [row for row in comparison_rows if row['mrr_at_10_gain'] > 0][:SAMPLE_SIZE]

display_metric_examples(
    recall_at_1_gain_rows, 'Recall@1 gains: correct answer moved to rank 1',
    'base_recall_at_1', 'fine_tuned_recall_at_1', 'recall_at_1_gain',
)
display_metric_examples(
    recall_at_5_gain_rows, 'Recall@5 gains: correct answer entered the top 5',
    'base_recall_at_5', 'fine_tuned_recall_at_5', 'recall_at_5_gain',
)
display_metric_examples(
    mrr_at_10_gain_rows, 'MRR@10 gains: reciprocal rank increased',
    'base_reciprocal_rank_at_10', 'fine_tuned_reciprocal_rank_at_10', 'mrr_at_10_gain', percent=True,
)

def retrieved_text(retrieval_record):
    return '\n\n'.join(
        f'#{item["rank"]} | score={item["score"]:.6f} | {item["answer_id"]}\n'
        + test_by_id[item['answer_id']]['content_text']
        for item in retrieval_record['retrieved']
    )

selected_for_review = {}
for row in recall_at_1_gain_rows + recall_at_5_gain_rows + mrr_at_10_gain_rows:
    selected_for_review[row['id']] = row

QUALITATIVE_OUTPUT_PATH = REPORT_DIR / 'qualitative_metric_improvements_base_vs_fine_tuned.csv'
with QUALITATIVE_OUTPUT_PATH.open('w', encoding='utf-8-sig', newline='') as target:
    writer = csv.DictWriter(target, fieldnames=[
        'id', 'question', 'correct_answer', 'base_rank', 'fine_tuned_rank',
        'recall_at_1_gain', 'recall_at_5_gain', 'mrr_at_10_gain',
        'base_top_5', 'fine_tuned_top_5',
    ])
    writer.writeheader()
    for row in selected_for_review.values():
        writer.writerow({
            **{field: row[field] for field in (
                'id', 'question', 'correct_answer', 'base_rank', 'fine_tuned_rank',
                'recall_at_1_gain', 'recall_at_5_gain', 'mrr_at_10_gain',
            )},
            'base_top_5': retrieved_text(base_retrieval_by_id[row['id']]),
            'fine_tuned_top_5': retrieved_text(fine_tuned_retrieval_by_id[row['id']]),
        })

print(f'Shareable qualitative examples: {QUALITATIVE_OUTPUT_PATH}')

## 6. Inspect a selected question across all available models

Set `SELECTED_QUERY_ID` to an ID from the previous tables. If Jina and Snowflake rank files exist, their ranks are shown too.

In [ ]:
SELECTED_QUERY_ID = strongest_improvements[0]['id'] if strongest_improvements else comparison_rows[0]['id']
OPTIONAL_MODEL_LABELS = ['jina_embeddings_v3', 'snowflake_arctic_embed_l_v2']

selected = next(row for row in comparison_rows if row['id'] == SELECTED_QUERY_ID)
model_ranks = [(BASE_LABEL, selected['base_rank']), (FINE_TUNED_LABEL, selected['fine_tuned_rank'])]

for label in OPTIONAL_MODEL_LABELS:
    path = REPORT_DIR / f'per_query_ranks_{label}.csv'
    if path.is_file():
        rows = read_csv_rows(path, rank_fields)
        rank = int(next(row['correct_answer_rank'] for row in rows if row['id'] == SELECTED_QUERY_ID))
        model_ranks.append((label, rank))

rank_lines = ''.join(
    f'<li><b>{html.escape(label)}</b>: rank {rank:,}</li>'
    for label, rank in model_ranks
)
base_retrieved_html = render_retrieved(SELECTED_QUERY_ID, base_retrieval_by_id[SELECTED_QUERY_ID])
fine_tuned_retrieved_html = render_retrieved(SELECTED_QUERY_ID, fine_tuned_retrieval_by_id[SELECTED_QUERY_ID])
display(HTML(
    f'<h4>{html.escape(selected["id"])}</h4>'
    f'<p><b>Question:</b> {html.escape(selected["question"])}</p>'
    f'<p><b>Correct answer:</b> {html.escape(selected["correct_answer"])}</p>'
    f'<ul>{rank_lines}</ul>'
    f'<h5>Base model top {RETRIEVED_TOP_K}</h5>{base_retrieved_html}'
    f'<h5>Fine-tuned model top {RETRIEVED_TOP_K}</h5>{fine_tuned_retrieved_html}'
))

## 7. Inspect regressions as a control

Reviewing the largest regressions prevents selective reporting and can reveal which question types require further data or model analysis.

In [ ]:
largest_regressions = sorted(
    regressed_rows,
    key=lambda row: (row['rank_improvement'], row['fine_tuned_rank'], row['id']),
)[:SAMPLE_SIZE]
display_examples(largest_regressions, 'Largest regressions after fine-tuning')